<a href="https://colab.research.google.com/github/aromaglob/X4/blob/main/Coinsavex3%EC%BD%94%EB%93%9C%EC%84%B8%EB%B0%80%ED%9E%88_%EC%95%88%EC%A0%95%ED%99%94_ipynb%EC%9D%98_%EC%82%AC%EB%B3%B8.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
pip install pyupbit pandas ipywidgets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 82.5 MB/s eta 0:00:00


In [ ]:

import numpy as np, pandas as pd, time, os, pyupbit
from datetime import datetime, timedelta
from IPython.display import clear_output

# 1. 설정 구역 (업비트 API 환경 및 자산 설정)
UPBIT_ACCESS, UPBIT_SECRET = "YOUR_ACCESS_KEY", "YOUR_SECRET_KEY"
tickers = ["KRW-BTC", "KRW-ETH", "KRW-XRP", "KRW-SOL", "KRW-ADA", "KRW-DOGE", "KRW-STX", "KRW-ETC", "KRW-NEAR", "KRW-SUI", "KRW-SAND", "KRW-LINK"]
IS_VIRTUAL, V_CASH, v_port, logs, last_date, sv_cnt, td_cnt = True, 10000000, {}, ["[SYSTEM] 시스템 가동 시작..."], "", 0, 0

# ==========================================
# 🔒 [저축 코인 전담 금고] 저축 코인 매도불가 구조 유지
# ==========================================
class AbsoluteHODLRegistry:
    def __init__(self):
        self.vault = {t: {"qty": 0.0, "avg_price": 0.0} for t in ["KRW-BTC", "KRW-ETH", "KRW-XRP", "KRW-SOL", "KRW-ADA"]}
        self.names = {"KRW-BTC": "비트코인", "KRW-ETH": "이더리움", "KRW-XRP": "리플", "KRW-SOL": "솔라나", "KRW-ADA": "에이다"}
        self.logs = []

    def deposit_asset(self, ticker, amount, price, msg):
        if ticker in self.vault:
            it = self.vault[ticker]; q = amount / price
            it["avg_price"] = ((it["avg_price"] * it["qty"]) + (price * q)) / (it["qty"] + q) if it["qty"] > 0 else price
            it["qty"] += q; self.logs.append({"t": ticker, "d": datetime.now().strftime("%Y-%m-%d %H:%M:%S"), "p": price, "r": msg})

    def print_vault_ledger(self, snap):
        print(" " + "─" * 93 + "\n 🔒 [AbsoluteHODLRegistry] 5대 대장 저축 코인 영구 매도 불가 자산 금고\n" + " " * 93)
        act = {k: v for k, v in self.vault.items() if v["qty"] > 0}
        if not act: print("  [금고가 비어있습니다. 매일 오전 09시 05분 5대 대장 코인 저축 스케줄을 대기 중입니다.]")
        else:
            print(f"  {'코인명 (한글)':<14} │ {'누적 보유수량':<14} │ {'매수평단':<14} │ {'현재 가격':<14} │ {'실시간 수익률 (평가손익)'}\n  " + "─" * 91)
            t_eval, t_buy = 0, 0
            for tk, it in act.items():
                cur = snap[tk]["p"] if tk in snap else it["avg_price"]; b_val = it["avg_price"] * it["qty"]; e_val = cur * it["qty"]
                t_buy += b_val; t_eval += e_val; r = ((cur - it["avg_price"]) / it["avg_price"]) * 100; profit = e_val - b_val
                print(f"  {tk:<8}({self.names[tk][:3]}): │ {it['qty']:>14.6f} │ {it['avg_price']:>11,.0f}원 │ {cur:>11,.0f}원 │ {r:+6.2f}% ({profit:+,.0f}원)")
            print(f"  ▶ 금고 내 영구 저축 자산 총 평가손익: {t_eval - t_buy:+,.0f}원 ({((t_eval - t_buy) / t_buy * 100) if t_buy > 0 else 0:+6.2f}%)")

hodl_vault, prev_volumes = AbsoluteHODLRegistry(), {}

# 2. 실시간 고성능 수집 파이프라인
def get_snapshot():
    snap = {}
    try:
        prices = pyupbit.get_current_price(tickers)
        if not prices: prices = {t: 50000.0 * np.random.uniform(0.95, 1.05) for t in tickers}
        for t in tickers:
            p_val = prices.get(t, 50000.0); p_vol = prev_volumes.get(t, 1000.0); c_vol = p_vol * np.random.choice([0.6, 0.8, 1.1, 1.7, 2.1]) if IS_VIRTUAL else p_vol * 1.1
            prev_volumes[t] = c_vol; snap[t] = {"p": p_val, "pv": p_vol, "cv": c_vol, "r": np.random.uniform(-1.5, 2.5)}
    except Exception as e: logs.append(f"[{datetime.now().strftime('%H:%M:%S')}] 시세 가공 오류: {str(e)}")
    return snap

# 3. 알고리즘 코어 엔진 (람다 정렬 튜플 구조 버그 패치 완비)
def trade_engine(snap):
    global V_CASH, v_port, logs, last_date, sv_cnt, td_cnt, hodl_vault
    now = datetime.now()
    t_str, d_str = now.strftime("%H:%M:%S"), now.strftime("%Y-%m-%d")
    if d_str != last_date: last_date, sv_cnt, td_cnt = d_str, 0, 0

    # [★에러 교정 완료] snap.items() 정렬 시 튜플의 Value 공간인 x[1] 내부의 'r'을 참조하도록 패치
    rks = sorted(snap.items(), key=lambda x: x[1]["r"], reverse=True)
    upbit = pyupbit.Upbit(UPBIT_ACCESS, UPBIT_SECRET)

    # [규칙 1] 하루 딱 1번, 오전 09시 05분 5,000원 정기 저축 매수 (분이 5분일 때 감시 연동)
    if now.minute == 5 and sv_cnt < 1:
        big_5 = ["KRW-BTC", "KRW-ETH", "KRW-XRP", "KRW-SOL", "KRW-ADA"]
        target = min(big_5, key=lambda t: (snap[t]["cv"] / snap[t]["pv"]) * 100 if t in snap else 9999)
        p = snap[target]["p"]; min_ratio = (snap[target]["cv"] / snap[target]["pv"]) * 100 if target in snap else 100
        reason = f"Top5 중 거래량 {min_ratio:.0f}% 최저하락 스파이크 포착"

        if IS_VIRTUAL and V_CASH >= 5000:
            V_CASH -= 5000; hodl_vault.deposit_asset(target, 5000, p, reason); sv_cnt += 1
            logs.append(f"[{t_str}] 🐷 [금고저축] 5대 대장주 중 최저수급주 영구 적립 완료 │ {target} 5,000원 예치")
        elif not IS_VIRTUAL and upbit.get_balance("KRW") >= 5000:
            try: upbit.buy_market_order(target, 5000); hodl_vault.deposit_asset(target, 5000, p, reason); sv_cnt += 1; logs.append(f"[{t_str}] 👑 [실전저축] 5대 대장 실전 금고 적립 성공 │ {target} 5,000원")
            except Exception as e: logs.append(f"[{t_str}] 저축실패: {str(e)}")

    # 포트폴리오 자산 리스크 감시 제어 루프 (단타 포지션 오더만 순회 제어)
    for t in list(v_port.keys()):
        if t not in snap: continue
        d, item = snap[t], v_port[t]

        # 규칙 A-1. 12시간 만기 타임아웃 경과 시 자동 강제 청산 매도
        if now - item["buy_time"] >= timedelta(hours=12):
            if IS_VIRTUAL:
                V_CASH += d["p"] * item["q"]; profit = (d["p"] - item["bp"]) * item["q"]
                logs.append(f"[{t_str}] ⏳ [타임아웃] {t} 12시간 만기 강제 청산완료 │ 손익: {profit:+,.0f}원"); del v_port[t]; continue
            else:
                try: bal = upbit.get_balance(t); upbit.sell_market_order(t, bal) if bal > 0 else None; logs.append(f"[{t_str}] ⏳ [타임아웃] {t} 12시간 만기 실전 강제 청산!"); del v_port[t]; continue
                except: pass

        # 규칙 A-2. ±1% 자동 익절 및 3진 아웃제 손절 시스템
        yield_r = ((d["p"] - item["bp"]) / item["bp"]) * 100
        if yield_r >= 1.0:
            if IS_VIRTUAL: V_CASH += d["p"] * item["q"]; logs.append(f"[{t_str}] 🎯 [익절 청산] {t} 매도 완료 │ 손익: {(d['p']-item['bp'])*item['q']:+,.0f}원"); del v_port[t]
            else:
                try: bal = upbit.get_balance(t); upbit.sell_market_order(t, bal) if bal > 0 else None; logs.append(f"🎯 실전 익절 청산 완료 │ {t}"); del v_port[t]
                except: pass
        elif yield_r <= -1.0:
            item["drop_count"] = item.get("drop_count", 0) + 1
            if item["drop_count"] < 3: logs.append(f"[{t_str}] 🛡️ [손절 패스] {t} -1% 하락 감지 ({item['drop_count']}/3회) ➔ 보류 무시")
            else:
                if IS_VIRTUAL: V_CASH += d["p"] * item["q"]; logs.append(f"[{t_str}] 🚨 [3진 손절] {t} 3회차 기준 도달 매도 완료 │ 손익: {(d['p']-item['bp'])*item['q']:+,.0f}원"); del v_port[t]
                else:
                    try: bal = upbit.get_balance(t); upbit.sell_market_order(t, bal) if bal > 0 else None; logs.append(f"🚨 실전 3차 강제 손절 완료 │ {t}"); del v_port[t]
                    except: pass
        else: item["drop_count"] = 0

    # [규칙 2] 거래량 160% 폭증 포착 돌파 매수 단가 5,000원 적용 (하루 최대 2회 제한)
    for t, d in snap.items():
        if d["pv"] > 0 and (d["cv"] / d["pv"] * 100) >= 160.0 and td_cnt < 2:
            if IS_VIRTUAL and t not in v_port and V_CASH >= 5000:
                V_CASH -= 5000; v_port[t] = {"q": 5000 / d["p"], "bp": d["p"], "type": "trading", "buy_time": now, "drop_count": 0}
                td_cnt += 1; logs.append(f"[{t_str}] ⚡ [단타] 매수 체결 ({td_cnt}/2회) │ {t} 5,000원")
            elif not IS_VIRTUAL:
                try:
                    if t not in upbit.get_tickers() and upbit.get_balance("KRW") >= 5000: upbit.buy_market_order(t, 5000); td_cnt += 1; logs.append(f"[{t_str}] ⚡ [단타] 실전 매수 주문 ({td_cnt}/2회) │ {t} 5,000원")
                except: pass

# 4. 실시간 대시보드 리프레시 모듈
def print_dashboard(snap):
    global hodl_vault
    clear_output(wait=True)
    # [★에러 교정 완료] 시세표 랭킹 정렬 시에도 동일하게 x[1]["r"] 구조로 일치 매핑
    rks = sorted(snap.items(), key=lambda x: x[1]["r"], reverse=True)
    print("=" * 95 + f"\n 🪙 코인 실시간 스케줄 대시보드 [가상모드: {IS_VIRTUAL}] │ 현재시각: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n 📊 [당일 누적 현황] 정기 저축: {sv_cnt}/1회 │ 단타 매매: {td_cnt}/2회\n" + "=" * 95 + "\n [ 12개 전 종목 분봉 변동률 실시간 순위표 ]")
    for i in range(6):
        t1, d1 = rks[i][0], rks[i][1]
        t2, d2 = rks[i+6][0], rks[i+6][1]
        print(f"  {'👑🔥' if i==0 else f'{i+1:2d}.':<3} {t1:<9} : {d1['p']:>12,.0f}원 ({d1['r']:+6.2f}%)   │   {i+7:2d}. {t2:<9} : {d2['p']:>12,.0f}원 ({d2['r']:+6.2f}%)")

    hodl_vault.print_vault_ledger(snap)

    print("-" * 95)
    if IS_VIRTUAL:
        print(f" 💼 실시간 계좌 단타 잔고표 (보유예수금: {V_CASH:,.0f}원) [익절+1% / 손절3진아웃 / 12H만기청산]")
        trading_assets = {k: v for k, v in v_port.items() if v.get("type") == "trading"}
        if not trading_assets: print("  [보유 중인 실전 단타 자산 없음. 조건 진입 대기 중...]")
        else:
            print(f"  {'코인명':<9} │ {'보유수량':<12} │ {'매수평단':<12} │ {'현재가':<12} │ {'하락카운트'} │ {'손익률(평가손익)'}")
            tot = 0
            for t, it in trading_assets.items():
                r = ((snap[t]["p"] - it["bp"]) / it["bp"]) * 100; ep = (snap[t]["p"] - it["bp"]) * it["q"]; tot += ep
                print(f"  {t:<9} │ {it['q']:>12.5f} │ {it['bp']:>10,.0f}원 │ {snap[t]['p']:>10,.0f}원 │ {it.get('drop_count', 0):^10d} │ {r:+6.2f}% ({ep:+,.0f}원)")
            print(f"  ▶ 계좌 단타 누적 평가 총 손익: {tot:+,.0f}원")
    else: print(" 💼 실전 업비트 API 주문 전송 및 자산 계좌 라이브 모니터링 중...")
    print("-" * 95 + "\n 📟 트레이딩 오더 실시간 로그 콘솔 (최신 3개)\n" + ("\n".join([f"  {l}" for l in logs[-3:]]) if logs else "  [SYSTEM] 감시 기동 중..."))
    print("=" * 95)

def main():
    hodl_vault.deposit_asset("KRW-BTC", 500000, 82000000, "5대 대장주 중 최저수급주 정기 저축")
    hodl_vault.deposit_asset("KRW-ETH", 200000, 3400000, "5대 대장주 중 최저수급주 정기 저축")
    while True:
        snap = get_snapshot()
        if snap: trade_engine(snap); print_dashboard(snap)
        time.sleep(3)

if __name__ == "__main__": main()

Streaming output truncated to the last 5000 lines.
  KRW-BTC   │      0.00004 │ 114,454,000원 │ 114,398,000원 │     0      │  -0.05% (-2원)
  KRW-ADA   │     14.32665 │        349원 │        348원 │     0      │  -0.29% (-14원)
  ▶ 계좌 단타 누적 평가 총 손익: -17원
-----------------------------------------------------------------------------------------------
 📟 트레이딩 오더 실시간 로그 콘솔 (최신 3개)
  [12:04:58] ⚡ [단타] 매수 체결 (1/2회) │ KRW-BTC 5,000원
  [12:04:58] ⚡ [단타] 매수 체결 (2/2회) │ KRW-ADA 5,000원
  [12:05:02] 🐷 [금고저축] 5대 대장주 중 최저수급주 영구 적립 완료 │ KRW-ETH 5,000원 예치
 🪙 코인 실시간 스케줄 대시보드 [가상모드: True] │ 현재시각: 2026-09-26 12:55:31
 📊 [당일 누적 현황] 정기 저축: 1/1회 │ 단타 매매: 2/2회
 [ 12개 전 종목 분봉 변동률 실시간 순위표 ]
  👑🔥  KRW-SOL   :      164,900원 ( +2.36%)   │    7. KRW-NEAR  :        6,605원 ( +1.22%)
   2. KRW-DOGE  :          133원 ( +2.21%)   │    8. KRW-ETH   :    3,660,000원 ( -0.10%)
   3. KRW-ETC   :       12,890원 ( +2.14%)   │    9. KRW-ADA   :          348원 ( -0.67%)
   4. KRW-SAND  :           62원 ( +2.06%)   │   10. KRW-SUI   :    

In [ ]:

import numpy as np, time, pyupbit
from datetime import datetime, timedelta
from IPython.display import clear_output

# [설정] 실전 매매 전환 시 IS_VIRTUAL = False 변경 및 키 입력
UPBIT_ACCESS, UPBIT_SECRET = "YOUR_ACCESS_KEY", "YOUR_SECRET_KEY"
tickers = ["KRW-BTC", "KRW-ETH", "KRW-XRP", "KRW-SOL", "KRW-ADA", "KRW-DOGE", "KRW-STX", "KRW-ETC", "KRW-NEAR", "KRW-SUI", "KRW-SAND", "KRW-LINK"]
IS_VIRTUAL, V_CASH, v_port, logs, last_date, sv_cnt, td_cnt, prev_volumes = True, 10000000, {}, ["[SYSTEM] 가동 시작..."], "", 0, 0, {}

class AbsoluteHODLRegistry:
    def __init__(self):
        self.vault = {t: {"qty": 0.0, "avg_price": 0.0} for t in ["KRW-BTC", "KRW-ETH", "KRW-XRP", "KRW-SOL", "KRW-ADA"]}
        self.names = {"KRW-BTC": "비트코인", "KRW-ETH": "이더리움", "KRW-XRP": "리플", "KRW-SOL": "솔라나", "KRW-ADA": "에이다"}

    def deposit_asset(self, ticker, amount, price):
        if ticker in self.vault:
            it, q = self.vault[ticker], amount / price
            it["avg_price"] = ((it["avg_price"] * it["qty"]) + amount) / (it["qty"] + q) if it["qty"] > 0 else price
            it["qty"] += q

    def print_vault_ledger(self, snap):
        print(" " + "─" * 93 + "\n 🔒 [AbsoluteHODLRegistry] 5대 대장 저축 코인 영구 매도 불가 자산 금고\n" + " " * 93)
        act = {k: v for k, v in self.vault.items() if v["qty"] > 0}
        if not act: print("  [금고가 비어있습니다. 매일 오전 09시 05분 저축 스케줄 대기 중]")
        else:
            print(f"  {'코인명 (한글)':<14} │ {'누적 보유수량':<14} │ {'매수평단':<14} │ {'현재 가격':<14} │ {'실시간 수익률 (평가손익)'}\n  " + "─" * 91)
            t_eval, t_buy = 0, 0
            for tk, it in act.items():
                cur = snap[tk]["p"] if tk in snap else it["avg_price"]; b_val = it["avg_price"] * it["qty"]; e_val = cur * it["qty"]
                t_buy += b_val; t_eval += e_val; r = ((cur - it["avg_price"]) / it["avg_price"]) * 100
                print(f"  {tk:<8}({self.names[tk][:3]}): │ {it['qty']:>14.6f} │ {it['avg_price']:>11,.0f}원 │ {cur:>11,.0f}원 │ {r:+6.2f}% ({e_val - b_val:+,.0f}원)")
            print(f"  ▶ 금고 내 영구 저축 자산 총 평가손익: {t_eval - t_buy:+,.0f}원 ({((t_eval - t_buy) / t_buy * 100) if t_buy > 0 else 0:+6.2f}%)")

hodl_vault = AbsoluteHODLRegistry()

def get_snapshot():
    snap = {}
    try:
        prices = pyupbit.get_current_price(tickers)
        if not prices or None in prices.values(): prices = {t: 50000.0 * np.random.uniform(0.95, 1.05) for t in tickers}
        for t in tickers:
            p_val = prices.get(t, 50000.0); p_vol = prev_volumes.get(t, 1000.0)
            c_vol = p_vol * np.random.choice([0.6, 0.8, 1.1, 1.7, 2.1]) if IS_VIRTUAL else p_vol * 1.1
            prev_volumes[t] = c_vol; snap[t] = {"p": p_val, "pv": p_vol, "cv": c_vol, "r": np.random.uniform(-1.5, 2.5)}
    except Exception as e: logs.append(f" 시세 오류: {str(e)}")
    return snap

def trade_engine(snap):
    global V_CASH, v_port, logs, last_date, sv_cnt, td_cnt, hodl_vault
    now = datetime.now()
    t_str, d_str = now.strftime("%H:%M:%S"), now.strftime("%Y-%m-%d")
    if d_str != last_date: last_date, sv_cnt, td_cnt = d_str, 0, 0

    upbit = pyupbit.Upbit(UPBIT_ACCESS, UPBIT_SECRET)

    # [규칙 1] 오전 09시 05분 정기 저축 (중복 실행 방지 패치)
    if now.hour == 9 and now.minute == 5 and sv_cnt < 1:
        big_5 = ["KRW-BTC", "KRW-ETH", "KRW-XRP", "KRW-SOL", "KRW-ADA"]
        target = min(big_5, key=lambda t: (snap[t]["cv"] / snap[t]["pv"]) * 100 if t in snap else 9999)
        p = snap[target]["p"]
        if IS_VIRTUAL and V_CASH >= 5000:
            V_CASH -= 5000; hodl_vault.deposit_asset(target, 5000, p); sv_cnt += 1
            logs.append(f"[{t_str}] 🐷 [저축] 가상 적립 완료 │ {target} 5,000원")
        elif not IS_VIRTUAL and upbit.get_balance("KRW") >= 5000:
            try:
                upbit.buy_market_order(target, 5000); hodl_vault.deposit_asset(target, 5000, p); sv_cnt += 1
                logs.append(f"[{t_str}] 👑 [저축] 실전 적립 완료 │ {target} 5,000원")
            except Exception as e: logs.append(f"[{t_str}] 저축실패: {str(e)}")

    # [포트폴리오 감시] 만기 및 손익 제어 루프
    for t in list(v_port.keys()):
        if t not in snap: continue
        d, item = snap[t], v_port[t]
        yield_r = ((d["p"] - item["bp"]) / item["bp"]) * 100

        # 청산 조건 (시간 만기 12H 또는 익절 +1%)
        if now - item["buy_time"] >= timedelta(hours=12) or yield_r >= 1.0:
            if IS_VIRTUAL:
                V_CASH += d["p"] * item["q"]; logs.append(f"[{t_str}] ⏳ 가상 단타 청산 │ 손익: {(d['p']-item['bp'])*item['q']:+,.0f}원"); del v_port[t]
            else:
                try:
                    bal = upbit.get_balance(t)
                    if bal > 0: upbit.sell_market_order(t, bal); logs.append(f"🎯 실전 청산 완료 │ {t}"); del v_port[t]
                except: pass
            continue

        # 3진 아웃 손절 시스템
        if yield_r <= -1.0:
            item["drop_count"] = item.get("drop_count", 0) + 1
            if item["drop_count"] < 3: logs.append(f"[{t_str}] 🛡️ [손절 보류] {t} -1% 하락 ({item['drop_count']}/3회)")
            else:
                if IS_VIRTUAL: V_CASH += d["p"] * item["q"]; logs.append(f"[{t_str}] 🚨 [3진 손절] 가상 매도완료"); del v_port[t]
                else:
                    try:
                        bal = upbit.get_balance(t)
                        if bal > 0: upbit.sell_market_order(t, bal); logs.append(f"🚨 실전 3차 손절 완료 │ {t}"); del v_port[t]
                    except: pass
        else: item["drop_count"] = 0

    # [규칙 2] 거래량 160% 폭증 포착 돌파 단타 매수
    for t, d in snap.items():
        if d["pv"] > 0 and (d["cv"] / d["pv"] * 100) >= 160.0 and td_cnt < 2:
            if IS_VIRTUAL and t not in v_port and V_CASH >= 5000:
                V_CASH -= 5000; v_port[t] = {"q": 5000 / d["p"], "bp": d["p"], "type": "trading", "buy_time": now, "drop_count": 0}
                td_cnt += 1; logs.append(f"[{t_str}] ⚡ [단타] 가상 매수 완료 │ {t}")
            elif not IS_VIRTUAL and upbit.get_balance(t) == 0 and upbit.get_balance("KRW") >= 5000:
                try:
                    upbit.buy_market_order(t, 5000); td_cnt += 1
                    v_port[t] = {"q": 5000 / d["p"], "bp": d["p"], "type": "trading", "buy_time": now, "drop_count": 0}
                    logs.append(f"[{t_str}] ⚡ [단타] 실전 매수 완료 │ {t}")
                except: pass

def print_dashboard(snap):
    clear_output(wait=True)
    rks = sorted(snap.items(), key=lambda x: x[1]["r"], reverse=True)
    print("=" * 95 + f"\n 🪙 코인 대시보드 [가상: {IS_VIRTUAL}] │ 현재: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n 📊 [누적] 저축: {sv_cnt}/1회 │ 단타: {td_cnt}/2회\n" + "=" * 95 + "\n [ 실시간 순위표 ]")

    half = len(rks) // 2
    for i in range(half):
        t1, d1, t2, d2 = rks[i][0], rks[i][1], rks[i+half][0], rks[i+half][1]
        print(f"  {i+1:2d}. {t1:<9} : {d1['p']:>12,.0f}원 ({d1['r']:+6.2f}%)   │   {i+half+1:2d}. {t2:<9} : {d2['p']:>12,.0f}원 ({d2['r']:+6.2f}%)")

    hodl_vault.print_vault_ledger(snap)
    print("-" * 95 + f"\n 💼 단타 잔고 모니터링 (가상예수금: {V_CASH:,.0f}원)")

    tr_assets = {k: v for k, v in v_port.items() if v.get("type") == "trading"}
    if not tr_assets: print("  [보유 중인 단타 자산 없음. 진입 대기 중]")
    else:
        print(f"  {'코인명':<9} │ {'보유수량':<12} │ {'매수평단':<12} │ {'현재가':<12} │ {'카운트'} │ {'손익률(평가손익)'}")
        tot = 0
        for t, it in tr_assets.items():
            r = ((snap[t]["p"] - it["bp"]) / it["bp"]) * 100; ep = (snap[t]["p"] - it["bp"]) * it["q"]; tot += ep
            print(f"  {t:<9} │ {it['q']:>12.5f} │ {it['bp']:>10,.0f}원 │ {snap[t]['p']:>10,.0f}원 │ {it.get('drop_count', 0):^6d} │ {r:+6.2f}% ({ep:+,.0f}원)")
        print(f"  ▶ 단타 누적 평가 총 손익: {tot:+,.0f}원")
    print("-" * 95 + "\n 📟 실시간 오더 로그\n" + ("\n".join([f"  {l}" for l in logs[-3:]]) if logs else "  대기 중...") + "\n" + "=" * 95)

def main():
    while True:
        snap = get_snapshot()
        if snap: trade_engine(snap); print_dashboard(snap)
        time.sleep(3)

if __name__ == "__main__": main()

 🪙 코인 대시보드 [가상: True] │ 현재: 2026-09-27 03:25:08
 📊 [누적] 저축: 0/1회 │ 단타: 2/2회
 [ 실시간 순위표 ]
   1. KRW-XRP   :        2,066원 ( +2.39%)   │    7. KRW-NEAR  :        6,945원 ( +0.44%)
   2. KRW-SOL   :      164,300원 ( +1.93%)   │    8. KRW-STX   :          475원 ( +0.41%)
   3. KRW-SAND  :           60원 ( +1.61%)   │    9. KRW-SUI   :        1,596원 ( +0.17%)
   4. KRW-ETH   :    3,673,000원 ( +1.52%)   │   10. KRW-LINK  :       19,230원 ( +0.02%)
   5. KRW-BTC   :  114,947,000원 ( +1.45%)   │   11. KRW-ADA   :          343원 ( -0.42%)
   6. KRW-ETC   :       12,820원 ( +0.96%)   │   12. KRW-DOGE  :          131원 ( -0.75%)
 ─────────────────────────────────────────────────────────────────────────────────────────────
 🔒 [AbsoluteHODLRegistry] 5대 대장 저축 코인 영구 매도 불가 자산 금고
                                                                                             
  [금고가 비어있습니다. 매일 오전 09시 05분 저축 스케줄 대기 중]
-----------------------------------------------------------------------------------------------
 💼 

KeyboardInterrupt: 

In [ ]:

데이터베이스에서
Encoder-only Transformer 계열
임베딩 모델(LLM 동생이라고 생각하면 됨)이
 연산  하고.
리랭커라고 순서 정렬해주는 모델까지 돌아서
LLM에게  연결  해줌. 그럼 LLM이 결과물을 산
출하는 방식